In [1]:
!pip install transformers torch matplotlib

In [16]:

import torch
import torch.nn.functional as F
from transformers import GPT2Tokenizer, GPT2Model

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2Model.from_pretrained("gpt2", output_hidden_states=True)
model.eval()

def cosine(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

def get_hidden(sentence):
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    return inputs, outputs.hidden_states[-1][0]

# Sentences
sent1 = "X defeated Y."
sent2 = "Y defeated X."

inputs1, h1 = get_hidden(sent1)
inputs2, h2 = get_hidden(sent2)

tokens1 = tokenizer.convert_ids_to_tokens(inputs1["input_ids"][0])
tokens2 = tokenizer.convert_ids_to_tokens(inputs2["input_ids"][0])

print("Tokens 1:", tokens1)
print("Tokens 2:", tokens2)

# X and Y are single tokens in GPT-2
idx_X1 = tokens1.index("X")
idx_Y1 = tokens1.index("ĠY")

idx_Y2 = tokens2.index("Y")
idx_X2 = tokens2.index("ĠX")

x1 = h1[idx_X1]
y1 = h1[idx_Y1]

y2 = h2[idx_Y2]
x2 = h2[idx_X2]

# embedding of " loser"
loser_id = tokenizer.encode(" loser")[0]
loser_vec = model.get_input_embeddings().weight[loser_id]

print("\nSentence 1: X defeated Y.")
print("X vs loser:", cosine(x1, loser_vec))
print("Y vs loser:", cosine(y1, loser_vec))

print("\nSentence 2: Y defeated X.")
print("X vs loser:", cosine(x2, loser_vec))
print("Y vs loser:", cosine(y2, loser_vec))

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens 1: ['X', 'Ġdefeated', 'ĠY', '.']
Tokens 2: ['Y', 'Ġdefeated', 'ĠX', '.']

Sentence 1: X defeated Y.
X vs loser: -0.13622839748859406
Y vs loser: -0.11179926246404648

Sentence 2: Y defeated X.
X vs loser: -0.11348851025104523
Y vs loser: -0.13342034816741943
